# Scanner Dev Evaluation

Evaluate a single scanner run from the `scanner_dev/ground_truth/` staging workflow.

Each scan mixes transcripts drawn from several `.eval` files (different benchmarks and/or methods). This notebook groups scan rows by their source `.eval` file, attaches manifest metadata (from `build/dev_t5_provenance.csv`), and compares scanner grades against per-eval-file target rules.

**Inputs:**
- `SCAN_RESULTS_PATH` — a `scan_id=*` directory produced by `scout scan scout.yaml`
- `PROVENANCE_CSV` — `build/dev_t5_provenance.csv` from `build_dataset.py`
- `VALIDATION_CSV` — the merged validation CSV referenced by `scout.yaml` (e.g. `dev_t5_validation.csv`)

**Target rules** are keyed by eval-file basename. Each rule is one of:
- `{"mode": "validation"}` — look up the per-transcript target in the merged validation CSV (e.g. human t5 labels)
- `{"mode": "uniform", "positive_rate": 1.0}` — assume every transcript is a violation (e.g. synthetic contamination / web-search runs)
- `{"mode": "uniform", "positive_rate": 0.0}` — assume no violations

Eval files not listed in `TARGET_RULES` still appear in descriptive plots but are skipped when computing performance metrics.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Reuse the shared loader from the main analysis package
_ANALYSIS_DIR = Path("../analysis").resolve()
if str(_ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(_ANALYSIS_DIR))

from scan_utils import load_scan_results  # noqa: E402

In [ ]:
import sys
from pathlib import Path

# Point this import at whichever scanner subdirectory you want to evaluate.
_CONFIG_DIR = Path("./ground_truth").resolve()

if str(_CONFIG_DIR) not in sys.path:
    sys.path.insert(0, str(_CONFIG_DIR))

from config import (  # noqa: E402
    EXCLUDE_EVAL_FILES,
    INCLUDE_EVAL_FILES,
    PROVENANCE_CSV,
    SCAN_RESULTS_PATH,
    SCANNER_KEY,
    TARGET_RULES,
    VALIDATION_CSV,
    VIOLATION_THRESHOLD,
)

## Load Data

Loads the scan parquet, attaches the `eval_file` basename to each row, merges manifest metadata from the provenance CSV, and loads the merged validation CSV.

In [ ]:
all_scans = load_scan_results(SCAN_RESULTS_PATH)
scans = all_scans[all_scans["scanner_key"] == SCANNER_KEY].copy()
if scans.empty:
    available = sorted(all_scans["scanner_key"].dropna().unique())
    raise ValueError(f"Scanner key '{SCANNER_KEY}' not found. Available: {available}")

scans["eval_file"] = scans["transcript_source_uri"].apply(
    lambda uri: Path(uri).name if isinstance(uri, str) else None
)

# Manifest / provenance metadata (Eval, method, validation_path, ...).
provenance = pd.read_csv(PROVENANCE_CSV)
provenance["eval_file"] = provenance["staged_eval_log_path"].apply(
    lambda p: Path(p).name if isinstance(p, str) else None
)
meta_cols = ["eval_file", "Eval", "method", "samples", "expected_v_rate",
             "validation_path", "include_in_validation"]
meta_cols = [c for c in meta_cols if c in provenance.columns]
scans = scans.merge(
    provenance[meta_cols].drop_duplicates(subset=["eval_file"]),
    on="eval_file",
    how="left",
)

# Benchmark group: collapses eval files that belong to the same benchmark.
# BENCHMARK_ALIASES merges task_set variants (e.g. the "_mini" subset maps
# back onto its parent benchmark).
BENCHMARK_ALIASES = {
    "swe_bench_verified_mini": "swe_bench",
}

def _benchmark_group(row: pd.Series) -> str | None:
    ts = row.get("transcript_task_set")
    if isinstance(ts, str) and ts:
        return BENCHMARK_ALIASES.get(ts, ts)
    ev = row.get("Eval")
    return str(ev) if pd.notna(ev) else None

scans["benchmark"] = scans.apply(_benchmark_group, axis=1)

# Apply optional filters.
if INCLUDE_EVAL_FILES:
    scans = scans[scans["eval_file"].isin(INCLUDE_EVAL_FILES)]
if EXCLUDE_EVAL_FILES:
    scans = scans[~scans["eval_file"].isin(EXCLUDE_EVAL_FILES)]

# Merged validation CSV.
validation = pd.read_csv(VALIDATION_CSV)
validation["target_num"] = pd.to_numeric(validation["target"], errors="coerce")
validated_ids = set(validation["id"].dropna())

print(f"Scanner: {SCANNER_KEY}")
print(f"Scan rows: {len(scans):,}")
print(f"Unique transcripts: {scans['transcript_id'].nunique():,}")
print(f"Unique eval files in scan: {scans['eval_file'].nunique():,}")
print(f"Unique benchmarks in scan: {scans['benchmark'].nunique():,}")
print(f"Validation CSV entries: {len(validation):,}")

In [ ]:
def _format_rule(rule: dict | None) -> str:
    if rule is None:
        return "—"
    mode = rule.get("mode")
    if mode == "validation":
        return "validation"
    if mode == "uniform":
        return f"uniform::{rule.get('positive_rate')}"
    return str(rule)


def _short_label(row: pd.Series) -> str:
    """Two-line plot label: Eval on top, method below."""
    eval_name = row.get("Eval") or row.get("transcript_task_set") or row["eval_file"]
    method = row.get("method")
    if method and pd.notna(method):
        return f"{eval_name}\n{method}"
    return str(eval_name)

In [ ]:
# Per-eval-file overview: manifest metadata, scan size, validation coverage,
# and the target rule (if any).
rows = []
for eval_file, group in scans.groupby("eval_file", dropna=False):
    n = len(group)
    n_validated = int(group["transcript_id"].isin(validated_ids).sum())
    rows.append({
        "eval_file": eval_file,
        "Eval": group["Eval"].iloc[0] if "Eval" in group.columns else None,
        "method": group["method"].iloc[0] if "method" in group.columns else None,
        "task_set": group["transcript_task_set"].iloc[0],
        "n_scanned": n,
        "n_validated": n_validated,
        "expected_v_rate": group["expected_v_rate"].iloc[0] if "expected_v_rate" in group.columns else None,
        "target_rule": _format_rule(TARGET_RULES.get(eval_file)),
    })

overview = pd.DataFrame(rows).sort_values(["Eval", "method", "eval_file"], na_position="last")
display(overview)

## Grade Distribution

Stacked grade distribution per benchmark × method (aggregating across repeated eval-file runs of the same benchmark).

In [ ]:
score_colors = {0: "#4393c3", 1: "#f4d35e", 2: "#d1495b", 3: "#7f0000"}

# Aggregate across eval files by (benchmark, method) so repeated runs of the
# same benchmark collapse into one bar per method.
group_keys = (
    scans.dropna(subset=["benchmark"])
    .groupby(["benchmark", "method"], dropna=False)
    .size()
    .sort_index()
    .index.tolist()
)

def _group_mask(key):
    bench, method = key
    m = scans["benchmark"] == bench
    if pd.isna(method):
        return m & scans["method"].isna()
    return m & (scans["method"] == method)

def _group_label(key):
    bench, method = key
    if pd.isna(method):
        return str(bench)
    return f"{bench}\n{method}"

group_labels = [_group_label(k) for k in group_keys]
sample_counts = [int(scans[_group_mask(k)]["value_num"].dropna().shape[0]) for k in group_keys]

grade_levels = sorted(scans["value_num"].dropna().astype(int).unique())

fig, ax = plt.subplots(figsize=(max(6, len(group_keys) * 1.5), 5))
x = np.arange(len(group_keys))
bottom = np.zeros(len(group_keys))

for grade in reversed(grade_levels):
    proportions = []
    for k in group_keys:
        subset_data = scans[_group_mask(k)]["value_num"].dropna()
        total = len(subset_data)
        proportions.append((subset_data.astype(int) == grade).sum() / total if total else 0)
    ax.bar(x, proportions, bottom=bottom, label=str(grade),
           color=score_colors.get(grade, "#999999"))
    bottom += np.array(proportions)

for xi, n in zip(x, sample_counts):
    ax.text(xi, .9, f"n={n}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(group_labels, rotation=45, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Proportion")
ax.set_title(f"Grade Distribution — {SCANNER_KEY}")
ax.legend(title="Grade")
fig.tight_layout()
plt.show()


## Detected Violation Rate per Eval File

Fraction of transcripts where the scanner score is `>= VIOLATION_THRESHOLD`. The reference line shows the target rate from `TARGET_RULES` (validation-mode rules use the mean positive rate in the matched validation rows).

In [ ]:
eval_files = sorted(scans["eval_file"].dropna().unique())

def _expected_rate(eval_file: str, group: pd.DataFrame) -> float | None:
    rule = TARGET_RULES.get(eval_file)
    if rule is None:
        return None
    if rule["mode"] == "uniform":
        return float(rule["positive_rate"])
    if rule["mode"] == "validation":
        ids = group["transcript_id"]
        matched = validation[validation["id"].isin(ids)]
        if matched.empty:
            return None
        return float(matched["target_num"].ge(VIOLATION_THRESHOLD).mean())
    return None


violation_rows = []
for ef in eval_files:
    group = scans[scans["eval_file"] == ef]
    scores = pd.to_numeric(group["value_num"], errors="coerce")
    n = int(scores.notna().sum())
    v_rate = float(scores.ge(VIOLATION_THRESHOLD).mean()) if n else np.nan
    violation_rows.append({
        "eval_file": ef,
        "label": _short_label(group.iloc[0]),
        "n": n,
        "detected_rate": v_rate,
        "expected_rate": _expected_rate(ef, group),
    })
violation_df = pd.DataFrame(violation_rows)

fig, ax = plt.subplots(figsize=(max(6, len(eval_files) * 1.2), 5))
xs = np.arange(len(violation_df))
bars = ax.bar(xs, violation_df["detected_rate"], color="#d1495b", label="Detected")

# Plot expected rate markers where a rule exists.
exp_mask = violation_df["expected_rate"].notna()
if exp_mask.any():
    ax.scatter(
        xs[exp_mask.values],
        violation_df.loc[exp_mask, "expected_rate"],
        marker="_", s=200, linewidths=3, color="#333333", label="Target",
    )

for i, (bar, n) in enumerate(zip(bars, violation_df["n"])):
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={n}",
            ha="center", va="bottom", fontsize=8, color="white")

ax.set_xticks(xs)
ax.set_xticklabels(violation_df["label"], rotation=45, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel(f"Violation rate (score ≥ {VIOLATION_THRESHOLD})")
ax.set_title(f"Detected Violation Rate — {SCANNER_KEY}")
ax.legend(loc="best")
fig.tight_layout()
plt.show()

display(violation_df[["label", "n", "detected_rate", "expected_rate"]])

## Performance vs Target Rules

For eval files listed in `TARGET_RULES`, computes accuracy, sensitivity (recall of positives) and specificity (recall of negatives) at the chosen violation threshold.

In [ ]:
# Collapse to one row per transcript per eval file, then aggregate metrics by
# (benchmark, method) so repeated eval files for the same benchmark+method
# collapse into a single row.
comparison = (
    scans.groupby(["eval_file", "transcript_id"], dropna=False)
    .agg(scanner_grade=("value_num", "first"),
         Eval=("Eval", "first"),
         method=("method", "first"),
         benchmark=("benchmark", "first"))
    .reset_index()
)

scanner_scores = pd.to_numeric(comparison["scanner_grade"], errors="coerce")
comparison["prediction"] = pd.Series(
    np.where(scanner_scores.isna(), pd.NA, scanner_scores.ge(VIOLATION_THRESHOLD)),
    index=comparison.index,
    dtype="boolean",
)
comparison["target"] = pd.Series(pd.NA, index=comparison.index, dtype="boolean")
comparison["target_grade"] = pd.NA
comparison["target_source"] = pd.NA

validation_lookup = validation.set_index("id")["target_num"]

for eval_file, rule in TARGET_RULES.items():
    mask = comparison["eval_file"] == eval_file
    if not mask.any():
        continue
    mode = rule.get("mode")
    if mode == "uniform":
        rate = rule.get("positive_rate")
        if rate not in {0, 0.0, 1, 1.0}:
            raise ValueError(
                f"Uniform rules only support 0.0 or 1.0 positive_rate; got {rate!r} for {eval_file}."
            )
        comparison.loc[mask, "target"] = bool(rate)
        comparison.loc[mask, "target_source"] = f"uniform::{rate}"
    elif mode == "validation":
        idx = comparison.index[mask]
        target_grade = comparison.loc[idx, "transcript_id"].map(validation_lookup)
        comparison.loc[idx, "target_grade"] = target_grade.values
        comparison.loc[idx, "target"] = pd.Series(
            np.where(target_grade.isna(), pd.NA, target_grade.ge(VIOLATION_THRESHOLD)),
            index=idx,
            dtype="boolean",
        )
        comparison.loc[idx, "target_source"] = "validation"
    else:
        raise ValueError(f"Unsupported rule mode {mode!r} for {eval_file}")

valid = comparison[comparison["target"].notna() & comparison["prediction"].notna()].copy()

if valid.empty:
    print("No transcript has both a scanner grade and a resolved target — check TARGET_RULES.")
else:
    metrics_rows = []
    for (benchmark, method), group in valid.groupby(["benchmark", "method"], dropna=False):
        pred = group["prediction"].astype(bool)
        tgt = group["target"].astype(bool)
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        n = len(group)
        sources = sorted(group["target_source"].dropna().unique())
        metrics_rows.append({
            "benchmark": benchmark,
            "method": method if pd.notna(method) else "—",
            "target_source": ", ".join(sources) if sources else "—",
            "n": n,
            "accuracy": (tp + tn) / n if n else np.nan,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        })
    metrics = pd.DataFrame(metrics_rows).sort_values(
        ["benchmark", "method"], na_position="last"
    )
    display_metrics = metrics.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        display_metrics[col] = display_metrics[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display(display_metrics)

## Disagreements

Per eval file, lists the false-positive and false-negative transcripts so you can open them for inspection.

In [ ]:
if valid.empty:
    print("No data with resolved targets — see above.")
else:
    detail_cols = ["transcript_id", "scanner_grade", "target_grade",
                   "prediction", "target", "target_source"]
    detail_cols = [c for c in detail_cols if c in valid.columns]

    for eval_file, group in valid.groupby("eval_file", dropna=False):
        label_row = scans[scans["eval_file"] == eval_file].iloc[0]
        label = _short_label(label_row).replace("\n", " / ")
        mismatches = group[group["prediction"].astype(bool) != group["target"].astype(bool)]

        print("=" * 70)
        print(f"{label}")
        print(f"  {eval_file}")
        print(f"  mismatches: {len(mismatches)} / {len(group)} "
              f"({(len(mismatches) / len(group)):.1%})")

        if mismatches.empty:
            print("  perfect agreement")
            continue

        fp = mismatches[mismatches["prediction"].astype(bool)]
        fn = mismatches[~mismatches["prediction"].astype(bool)]
        print(f"  false positives (flagged, target clean): {len(fp)}")
        if not fp.empty:
            display(fp[detail_cols].reset_index(drop=True))
        print(f"  false negatives (missed, target violation): {len(fn)}")
        if not fn.empty:
            display(fn[detail_cols].reset_index(drop=True))

## Confusion Matrices vs Human Labels

For each eval file whose target rule is `validation` (human-labeled), plots a 4x4 confusion matrix (rows = human grade, cols = scanner grade) over grades 0–3, and reports the quadratic-weighted Cohen's kappa.

In [ ]:
GRADE_LEVELS = [0, 1, 2, 3]


def _confusion_matrix(human: np.ndarray, scanner: np.ndarray, levels: list[int]) -> np.ndarray:
    """Fixed-size confusion matrix; rows = human, cols = scanner."""
    idx = {g: i for i, g in enumerate(levels)}
    k = len(levels)
    cm = np.zeros((k, k), dtype=int)
    for h, s in zip(human, scanner):
        if h in idx and s in idx:
            cm[idx[h], idx[s]] += 1
    return cm


def _quadratic_weighted_kappa(cm: np.ndarray) -> float:
    """Quadratic-weighted Cohen's kappa from a square confusion matrix."""
    n = cm.sum()
    if n == 0:
        return float("nan")
    k = cm.shape[0]
    if k < 2:
        return float("nan")
    weights = (np.arange(k)[:, None] - np.arange(k)[None, :]) ** 2 / (k - 1) ** 2
    observed = cm / n
    row_marg = cm.sum(axis=1) / n
    col_marg = cm.sum(axis=0) / n
    expected = np.outer(row_marg, col_marg)
    denom = (weights * expected).sum()
    if denom == 0:
        return float("nan")
    return 1.0 - (weights * observed).sum() / denom


# Aggregate human-labeled runs by benchmark so repeated eval files for the
# same benchmark (e.g. swe_bench + swe_bench_verified_mini, or two mle_bench
# runs) collapse into a single confusion matrix.
validation_files = [
    ef for ef, rule in TARGET_RULES.items()
    if rule.get("mode") == "validation" and (scans["eval_file"] == ef).any()
]

if not validation_files:
    print("No eval files with validation-mode targets are present in this scan.")
else:
    eval_to_benchmark = (
        scans.drop_duplicates("eval_file")
        .set_index("eval_file")["benchmark"]
        .to_dict()
    )
    validation_comparison = comparison[comparison["eval_file"].isin(validation_files)].copy()
    validation_comparison["benchmark"] = validation_comparison["eval_file"].map(eval_to_benchmark)
    validation_benchmarks = sorted(validation_comparison["benchmark"].dropna().unique())

    n_files = len(validation_benchmarks)
    ncols = min(2, n_files)
    nrows = int(np.ceil(n_files / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.0 * nrows), squeeze=False)

    kappa_rows = []
    for ax_idx, benchmark in enumerate(validation_benchmarks):
        ax = axes[ax_idx // ncols][ax_idx % ncols]
        group = validation_comparison[validation_comparison["benchmark"] == benchmark].copy()
        group["human_grade"] = pd.to_numeric(group["target_grade"], errors="coerce")
        group["scanner_grade_num"] = pd.to_numeric(group["scanner_grade"], errors="coerce")
        paired = group.dropna(subset=["human_grade", "scanner_grade_num"])

        human = paired["human_grade"].astype(int).to_numpy()
        scanner = paired["scanner_grade_num"].astype(int).to_numpy()
        cm = _confusion_matrix(human, scanner, GRADE_LEVELS)
        kappa = _quadratic_weighted_kappa(cm)

        im = ax.imshow(cm, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(GRADE_LEVELS)))
        ax.set_yticks(range(len(GRADE_LEVELS)))
        ax.set_xticklabels(GRADE_LEVELS)
        ax.set_yticklabels(GRADE_LEVELS)
        ax.set_xlabel("Scanner grade")
        ax.set_ylabel("Human grade")
        vmax = cm.max() if cm.max() > 0 else 1
        for i in range(len(GRADE_LEVELS)):
            for j in range(len(GRADE_LEVELS)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > vmax / 2 else "black", fontsize=9)
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        ax.set_title(f"{benchmark}\nn={len(paired)}, qwκ={kappa_str}", fontsize=9)
        kappa_rows.append({
            "benchmark": benchmark,
            "n": len(paired),
            "quadratic_weighted_kappa": kappa,
        })

    for extra in range(n_files, nrows * ncols):
        axes[extra // ncols][extra % ncols].axis("off")

    fig.suptitle(f"Confusion Matrices vs Human Labels — {SCANNER_KEY}", y=0.99)
    fig.tight_layout()
    plt.show()

    kappa_df = pd.DataFrame(kappa_rows)
    display_kappa = kappa_df.copy()
    display_kappa["quadratic_weighted_kappa"] = display_kappa["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display(display_kappa)

## Combined Across All Evals

Aggregates across every eval file with a resolved target. The metrics table pools all transcripts into a single row; the confusion matrix pools every validation-mode (human-labeled) transcript into a single 4×4 (scanner vs human grade).

In [ ]:
if valid.empty:
    print("No data with resolved targets — nothing to aggregate.")
else:
    # Pooled metrics across every eval with a resolved target.
    pred_all = valid["prediction"].astype(bool)
    tgt_all = valid["target"].astype(bool)
    tp = int((pred_all & tgt_all).sum())
    tn = int((~pred_all & ~tgt_all).sum())
    fp = int((pred_all & ~tgt_all).sum())
    fn = int((~pred_all & tgt_all).sum())
    n = len(valid)
    sources = sorted(valid["target_source"].dropna().unique())
    combined_metrics = pd.DataFrame([{
        "scope": "all evals combined",
        "target_source": ", ".join(sources) if sources else "—",
        "n": n,
        "accuracy": (tp + tn) / n if n else np.nan,
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }])
    display_combined = combined_metrics.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        display_combined[col] = display_combined[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display(display_combined)

    # Pooled confusion matrix across validation-mode rows only (human grades
    # exist only there).
    validation_rows = valid[valid["target_source"] == "validation"].copy()
    validation_rows["human_grade"] = pd.to_numeric(validation_rows["target_grade"], errors="coerce")
    validation_rows["scanner_grade_num"] = pd.to_numeric(validation_rows["scanner_grade"], errors="coerce")
    paired = validation_rows.dropna(subset=["human_grade", "scanner_grade_num"])

    if paired.empty:
        print("No validation-mode rows — skipping combined confusion matrix.")
    else:
        human = paired["human_grade"].astype(int).to_numpy()
        scanner = paired["scanner_grade_num"].astype(int).to_numpy()
        cm = _confusion_matrix(human, scanner, GRADE_LEVELS)
        kappa = _quadratic_weighted_kappa(cm)

        fig, ax = plt.subplots(figsize=(4.5, 4.2))
        im = ax.imshow(cm, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(GRADE_LEVELS)))
        ax.set_yticks(range(len(GRADE_LEVELS)))
        ax.set_xticklabels(GRADE_LEVELS)
        ax.set_yticklabels(GRADE_LEVELS)
        ax.set_xlabel("Scanner grade")
        ax.set_ylabel("Human grade")
        vmax = cm.max() if cm.max() > 0 else 1
        for i in range(len(GRADE_LEVELS)):
            for j in range(len(GRADE_LEVELS)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > vmax / 2 else "black", fontsize=9)
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        ax.set_title(f"All validation evals combined\nn={len(paired)}, qwκ={kappa_str}", fontsize=10)
        fig.tight_layout()
        plt.show()